In [3]:
# UI-driven single cell: choose files and run the full conversion
import numpy as np
import pandas as pd
import chess
from tqdm import tqdm
from IPython.display import display, clear_output
import ipywidgets as widgets
import os

# --- Conversion helper functions (self-contained) ---
def board_to_matrix(board, piece_type):
    piece_map = board.piece_map()
    matrix = np.zeros((8,8))
    for square, piece in piece_map.items():
        rank = chess.square_rank(square)
        file = chess.square_file(square)
        if piece.piece_type == piece_type:
            matrix[7 - rank, file] = (-1 if piece.color == chess.BLACK else 1) * 1
    return matrix

def board2rep(board):
    pieces = [1,2,3,4,5,6]
    layers = []
    for piece in pieces:
        matrix = board_to_matrix(board, piece)
        white_matrix = np.where(matrix == 1, 1, 0)
        black_matrix = np.where(matrix == -1, 1, 0)
        layers.append(white_matrix)
        layers.append(black_matrix)
    if board.turn:
        color_matrix = np.ones((8,8))
        layers.append(color_matrix)
    else:
        color_matrix = np.zeros((8,8))
        layers.append(color_matrix)
    board_rep = np.stack(layers)
    board_rep = board_rep.astype(bool)
    return board_rep

# Movement codes: 64 possible move-types x origin-square (8x8)
codes, i = {}, 0
for nSquares in range(1,8):
    for direction in [(0,1), (1,1), (1,0), (1,-1), (0,-1), (-1,-1), (-1,0), (-1,1)]:
        codes[(nSquares*direction[0], nSquares*direction[1])] = i
        i += 1
# Knight moves (8)
for km in [(1,2),(2,1),(2,-1),(1,-2),(-1,-2),(-2,-1),(-2,1),(-1,2)]:
    codes[km] = i
    i += 1

def legal_moves_to_64_8_8(board):
    legal_moves = list(board.legal_moves)
    extracted_moves = [[ [chess.square_rank(m.from_square), chess.square_file(m.from_square)],
                         (chess.square_file(m.to_square) - chess.square_file(m.from_square), chess.square_rank(m.to_square) - chess.square_rank(m.from_square)) ] for m in legal_moves]
    array6488 = np.zeros((64,8,8))
    for em in extracted_moves:
        move_code = codes.get(tuple(em[1]))
        if move_code is None:
            continue
        array6488[move_code, 7-em[0][0], em[0][1]] = 1
    array6488 = array6488.astype(bool)
    return array6488

def uci_to_number(uci_move):
    m = chess.Move.from_uci(uci_move)
    move_code = codes[(chess.square_file(m.to_square) - chess.square_file(m.from_square), chess.square_rank(m.to_square) - chess.square_rank(m.from_square))]
    pos = np.ravel_multi_index(multi_index=((move_code, 7-chess.square_rank(m.from_square), chess.square_file(m.from_square))), dims=(64,8,8))
    return pos

def concat_fen_legal(fen):
    board = chess.Board(fen)
    fen_matrix = board2rep(board)
    legal_moves = legal_moves_to_64_8_8(board)
    fen_matrix_legal_moves = np.concatenate((fen_matrix, legal_moves), 0)
    fen_matrix_legal_moves = fen_matrix_legal_moves.reshape(-1)
    return fen_matrix_legal_moves

def normalize_perspective(fen, move_uci):
    board = chess.Board(fen)
    if board.turn == chess.BLACK:
        flipped_board = board.mirror()
        move = chess.Move.from_uci(move_uci)
        flipped_move = chess.Move(chess.square_mirror(move.from_square), chess.square_mirror(move.to_square), promotion=move.promotion)
        return flipped_board.fen(), flipped_move.uci()
    return fen, move_uci

def converter(df_mates, file_size=1000):
    dfs = []
    for start in tqdm(range(0, len(df_mates), file_size)):
        chunk = df_mates.iloc[start:start + file_size].copy()
        chunk[['fen', 'move']] = chunk.apply(lambda r: pd.Series(normalize_perspective(r['fen'], r['move'])), axis=1)
        df_chunk = pd.DataFrame({
            'board': chunk['fen'].apply(concat_fen_legal),
            'best': chunk['move'].apply(uci_to_number),
            'fen_original': chunk['fen'],
            'best_uci': chunk['move'],
            'value': chunk['value']
        })
        dfs.append(df_chunk)
    return pd.concat(dfs, ignore_index=True)

def load_input_file(filepath, input_format):
    """Load data from JSON or CSV file"""
    print(f'Loading {filepath} as {input_format.upper()}...')
    try:
        if input_format.lower() == 'json':
            df = pd.read_json(filepath)
        elif input_format.lower() == 'csv':
            df = pd.read_csv(filepath)
        else:
            raise ValueError(f'Unsupported input format: {input_format}')
        print(f'Loaded {len(df)} rows')
        return df
    except Exception as e:
        raise Exception(f'Error loading input file: {e}')

def save_output_file(df, filepath, output_format):
    """Save data to PARQUET or CSV file"""
    print(f'Saving to {filepath} as {output_format.upper()}...')
    try:
        if output_format.lower() == 'parquet':
            df.to_parquet(filepath, compression='gzip')
        elif output_format.lower() == 'csv':
            df.to_csv(filepath, index=False)
        else:
            raise ValueError(f'Unsupported output format: {output_format}')
        print(f'Saved to {filepath}')
    except Exception as e:
        raise Exception(f'Error saving output file: {e}')

# --- UI widgets ---
input_file = widgets.Text(value='dataset_all.json', description='Input file:')
input_format = widgets.Dropdown(options=['json', 'csv'], value='json', description='Input format:')
output_file = widgets.Text(value='lc0_converted.parquet.gzip', description='Output file:')
output_format = widgets.Dropdown(options=['parquet', 'csv'], value='parquet', description='Output format:')
chunk_size = widgets.IntText(value=1000, description='Chunk size:')
run_btn = widgets.Button(description='Run conversion', button_style='success')
status = widgets.Output()

def on_run(b):
    with status:
        clear_output()
        try:
            # Load input file
            df = load_input_file(input_file.value, input_format.value)
            
            # Normalize value if present
            if 'value' in df.columns:
                df['value'] = (df['value'] / 152.65).clip(-1, 1)
                print('Normalized value column')
            
            # Run conversion
            print('Starting conversion (this can take a while)...')
            df_conv = converter(df, file_size=int(chunk_size.value))
            
            # Save output file
            save_output_file(df_conv, output_file.value, output_format.value)
            print('✓ Conversion completed successfully!')
            
        except Exception as e:
            print(f'❌ Error: {e}')

run_btn.on_click(on_run)
ui = widgets.VBox([
    widgets.HBox([input_file, input_format]),
    widgets.HBox([output_file, output_format]),
    chunk_size,
    run_btn,
    status
])
display(ui)

In [2]:
# Technical visualization cell (hidden). The main UI handles running the conversion.

In [3]:
# Technical helper cell (hidden). All helpers are included in the UI cell to run everything from one place.